# Phase 3C — RAG Generation Evaluation

This notebook evaluates two generation approaches using the same Phase 3A
ground-truth dataset and retrieved source chunks.

Approaches:

1. Baseline grounded prompt using `gpt-4o-mini`
2. Improved grounded prompt using `gpt-4o-mini`

Evaluation:

- LLM-as-a-judge classification:
  - RELEVANT
  - PARTLY_RELEVANT
  - NON_RELEVANT
- Input, output, and total tokens
- Generation latency
- Estimated cost
- Judge reasoning

The ground-truth source chunk is included in the judge prompt as the reference
answer context.

In [1]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

GROUND_TRUTH_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "ground-truth.json"
)

CHUNKS_PATH = PROJECT_ROOT / "data" / "chunks.parquet"

print("Project root:", PROJECT_ROOT)
print("Ground truth exists:", GROUND_TRUTH_PATH.exists())
print("Chunks file exists:", CHUNKS_PATH.exists())

Project root: /workspaces/pm-playbook
Ground truth exists: True
Chunks file exists: True


In [2]:
## Load and validate the evaluation data

with GROUND_TRUTH_PATH.open(encoding="utf-8") as f:
    ground_truth = json.load(f)

ground_truth_df = pd.DataFrame(ground_truth)
chunks_df = pd.read_parquet(CHUNKS_PATH)

print("Ground-truth shape:", ground_truth_df.shape)
print("Chunks shape:", chunks_df.shape)

required_columns = {
    "question_id",
    "question",
    "expected_answer",
    "chunk_id",
    "source_text",
}

print(
    "Required ground-truth columns present:",
    required_columns.issubset(ground_truth_df.columns),
)

print(
    "Unique questions:",
    ground_truth_df["question"].nunique(),
)

print(
    "Unique chunk IDs:",
    ground_truth_df["chunk_id"].nunique(),
)

missing_chunk_ids = (
    set(ground_truth_df["chunk_id"])
    - set(chunks_df["chunk_id"])
)

print("Missing source chunks:", len(missing_chunk_ids))

Ground-truth shape: (180, 19)
Chunks shape: (50910, 15)
Required ground-truth columns present: True
Unique questions: 180
Unique chunk IDs: 180
Missing source chunks: 0


In [3]:
## Create a small evaluation sample

EVALUATION_SAMPLE_SIZE = 5
EVALUATION_RANDOM_SEED = 42

evaluation_sample_df = (
    ground_truth_df
    .sample(
        n=EVALUATION_SAMPLE_SIZE,
        random_state=EVALUATION_RANDOM_SEED,
    )
    .reset_index(drop=True)
)

print("Sample size:", len(evaluation_sample_df))

evaluation_sample_df[
    [
        "question_id",
        "question",
        "guest",
        "topic",
        "word_count",
    ]
]

Sample size: 5


,question_id,question,guest,topic,word_count
0,question-94f41cc14cb35796d01bf310f052bda1,How can individuals in product management ensu...,John Cutler,career development,101
1,question-bc48db169054dbcdafcd1d6fbccb6702,What is the minimum number of users suggested ...,Ronny Kohavi,A/B Testing,66
2,question-a03266ce0d5eb8986e288a1069e4348b,What is a valuable question to ask candidates ...,Ken Norton,interviewing,106
3,question-98c62cabd4f4b0e889e1ba5d8e3423f1,What factors are contributing to a potential i...,Julia Schottenstein,M&A,144
4,question-a20e2f18c0839f18c81e91b9d4c0df95,How can you effectively respond to a question ...,Wes Kao,communication,136


In [4]:
import sys

project_root_str = str(PROJECT_ROOT)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print("Project root added to sys.path:", sys.path[0])

Project root added to sys.path: /workspaces/pm-playbook


In [5]:
## Import and initialize the baseline RAG service

from pm_playbook.rag import (
    PMPlaybookRAG,
    SYSTEM_INSTRUCTIONS,
)

baseline_rag = PMPlaybookRAG(
    model="gpt-4o-mini",
    num_results=5,
    chunks_path=CHUNKS_PATH,
)

print("Baseline model:", baseline_rag.model)
print("Chunks path:", CHUNKS_PATH)
print("Chunks path exists:", CHUNKS_PATH.exists())

Baseline model: gpt-4o-mini
Chunks path: /workspaces/pm-playbook/data/chunks.parquet
Chunks path exists: True


In [6]:
## Run one baseline generation

sample_row = evaluation_sample_df.iloc[0]

baseline_result = baseline_rag.answer(
    question=sample_row["question"],
    num_results=5,
)

print("Question:")
print(sample_row["question"])

print("\nGenerated answer:")
print(baseline_result.answer)

print("\nExpected answer:")
print(sample_row["expected_answer"])

print("\nMetadata:")
print("Model:", baseline_result.model)
print("Sources:", baseline_result.retrieval_results)
print("Input tokens:", baseline_result.input_tokens)
print("Output tokens:", baseline_result.output_tokens)
print("Total tokens:", baseline_result.total_tokens)
print("Latency ms:", baseline_result.latency_ms)
print("Response ID:", baseline_result.response_id)

Question:
How can individuals in product management ensure they are making progress in their careers despite potential limitations at their current company?

Generated answer:
To make progress in product management careers despite potential limitations at their current company, individuals should focus on understanding and navigating the constraints of their environment. Matt LeMay emphasizes that recognizing and working within these constraints can provide a significant commercial advantage, particularly in B2B contexts, where the connections to customers and their needs are clearer and more actionable [Source 1].

Additionally, effective product managers should continuously seek to comprehend what success means for their specific business context. This involves being curious about various success metrics influenced by factors like the company's funding model, business model, and stakeholder expectations [Source 2]. For instance, understanding what investors want—whether it's growth, 

In [7]:
def build_oracle_context(record: pd.Series) -> str:
    """
    Build evaluation context from the exact ground-truth source chunk.
    """
    return (
        "[Source 1]\n"
        f"Episode guest: {record['guest']}\n"
        f"Episode: {record['episode_title']}\n"
        f"Actual speaker: {record['speaker_name']}\n"
        "Speaker role: episode guest\n"
        f"Excerpt:\n{record['source_text']}"
    )


oracle_context = build_oracle_context(sample_row)

print("Question:")
print(sample_row["question"])

print("\nOracle context:")
print(oracle_context)

print("\nExpected answer:")
print(sample_row["expected_answer"])

Question:
How can individuals in product management ensure they are making progress in their careers despite potential limitations at their current company?

Oracle context:
[Source 1]
Episode guest: John Cutler
Episode: What differentiates the highest-performing product teams | John Cutler (The Beautiful Mess)
Actual speaker: John Cutler
Speaker role: episode guest
Excerpt:
But do you have an opportunity to kind of nudge things forward in your space? Probably. And bring to the systems thing, is it true that some people don't have the privilege of leaving their company for whatever reason? That is true also. So many things can be true, but that doesn't mean that you can't try, I think to kind of almost write your portfolio as you go. Because if you wait two years, you're going to think it was just all a blur and messed up. Whereas the opportunities might be there in your day-to-day as you're working through.

Expected answer:
Individuals can write their portfolio as they go and look fo

In [8]:
##  Create an oracle-context generation function

from time import perf_counter
from typing import Any


def extract_response_usage(response: Any) -> tuple[int, int, int]:
    """Extract token usage from an OpenAI Responses API result."""
    usage = getattr(response, "usage", None)

    if usage is None:
        return 0, 0, 0

    input_tokens = int(getattr(usage, "input_tokens", 0) or 0)
    output_tokens = int(getattr(usage, "output_tokens", 0) or 0)
    total_tokens = int(
        getattr(
            usage,
            "total_tokens",
            input_tokens + output_tokens,
        )
        or input_tokens + output_tokens
    )

    return input_tokens, output_tokens, total_tokens


def generate_with_oracle_context(
    record: pd.Series,
    *,
    instructions: str,
    prompt_variant: str,
    model: str = "gpt-4o-mini",
) -> dict:
    """
    Generate an answer using the exact Phase 3A ground-truth source chunk.
    """
    context = build_oracle_context(record)

    user_input = (
        "Transcript excerpt:\n\n"
        f"{context}\n\n"
        "User question:\n"
        f"{record['question']}\n\n"
        "Provide a grounded answer with an inline [Source 1] citation."
    )

    started_at = perf_counter()

    response = baseline_rag.client.responses.create(
        model=model,
        instructions=instructions,
        input=user_input,
    )

    latency_ms = (perf_counter() - started_at) * 1000

    input_tokens, output_tokens, total_tokens = extract_response_usage(
        response
    )

    answer = response.output_text.strip()

    if not answer:
        raise RuntimeError("OpenAI returned an empty answer.")

    return {
        "question_id": record["question_id"],
        "question": record["question"],
        "expected_answer": record["expected_answer"],
        "source_text": record["source_text"],
        "chunk_id": record["chunk_id"],
        "guest": record["guest"],
        "prompt_variant": prompt_variant,
        "model": model,
        "answer": answer,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "latency_ms": latency_ms,
        "response_id": getattr(response, "id", None),
    }

In [9]:
baseline_oracle_result = generate_with_oracle_context(
    sample_row,
    instructions=SYSTEM_INSTRUCTIONS,
    prompt_variant="baseline_prompt",
)

print("Generated answer:")
print(baseline_oracle_result["answer"])

print("\nExpected answer:")
print(baseline_oracle_result["expected_answer"])

print("\nMetadata:")
print("Input tokens:", baseline_oracle_result["input_tokens"])
print("Output tokens:", baseline_oracle_result["output_tokens"])
print("Total tokens:", baseline_oracle_result["total_tokens"])
print(
    "Latency ms:",
    round(baseline_oracle_result["latency_ms"], 2),
)

Generated answer:
John Cutler suggests that individuals in product management can focus on finding opportunities within their current roles to nudge things forward. He emphasizes the importance of taking proactive steps to document and grow your portfolio over time. Instead of viewing career advancement as something that solely relies on changing companies, he encourages making the most of day-to-day opportunities. This approach can help avoid feelings of stagnation and ensure that you accumulate valuable experiences and achievements in your current position [Source 1].

Expected answer:
Individuals can write their portfolio as they go and look for opportunities in their day-to-day work to nudge things forward in their space.

Metadata:
Input tokens: 450
Output tokens: 94
Total tokens: 544
Latency ms: 3019.64


In [10]:
IMPROVED_SYSTEM_INSTRUCTIONS = """
You are PM Playbook, a product-management assistant grounded in
transcripts from Lenny's Podcast.

Answer the user's question using only the supplied transcript excerpt.

Requirements:
- Give the direct answer first.
- Preserve the source's specific advice and terminology.
- Include only claims directly supported by the excerpt.
- Do not add broader interpretations, motivations, outcomes, or recommendations
  that are not explicitly supported.
- Attribute advice to the actual speaker when helpful.
- Cite the answer with [Source 1].
- Use one short paragraph, usually 2 to 4 sentences.
- Do not quote unless the exact wording is important.
- If the excerpt does not answer the question, say so clearly.
""".strip()

print("Baseline prompt length:", len(SYSTEM_INSTRUCTIONS))
print("Improved prompt length:", len(IMPROVED_SYSTEM_INSTRUCTIONS))

Baseline prompt length: 1190
Improved prompt length: 707


In [11]:
improved_oracle_result = generate_with_oracle_context(
    sample_row,
    instructions=IMPROVED_SYSTEM_INSTRUCTIONS,
    prompt_variant="improved_concise_prompt",
)

print("Improved generated answer:")
print(improved_oracle_result["answer"])

print("\nBaseline generated answer:")
print(baseline_oracle_result["answer"])

print("\nExpected answer:")
print(improved_oracle_result["expected_answer"])

print("\nImproved metadata:")
print("Input tokens:", improved_oracle_result["input_tokens"])
print("Output tokens:", improved_oracle_result["output_tokens"])
print("Total tokens:", improved_oracle_result["total_tokens"])
print(
    "Latency ms:",
    round(improved_oracle_result["latency_ms"], 2),
)

Improved generated answer:
Individuals in product management can ensure they are making progress by looking for opportunities to "nudge things forward" within their current roles, as suggested by John Cutler. He emphasizes the importance of actively engaging with daily tasks to "almost write your portfolio as you go," rather than waiting for ideal circumstances or a future timeframe to assess their progress. This proactive approach can help them leverage available opportunities even if they feel limited by their current situation [Source 1].

Baseline generated answer:
John Cutler suggests that individuals in product management can focus on finding opportunities within their current roles to nudge things forward. He emphasizes the importance of taking proactive steps to document and grow your portfolio over time. Instead of viewing career advancement as something that solely relies on changing companies, he encourages making the most of day-to-day opportunities. This approach can help 

In [12]:
sample_generation_rows = []

for _, record in evaluation_sample_df.iterrows():
    baseline_output = generate_with_oracle_context(
        record,
        instructions=SYSTEM_INSTRUCTIONS,
        prompt_variant="baseline_prompt",
    )
    sample_generation_rows.append(baseline_output)

    improved_output = generate_with_oracle_context(
        record,
        instructions=IMPROVED_SYSTEM_INSTRUCTIONS,
        prompt_variant="improved_concise_prompt",
    )
    sample_generation_rows.append(improved_output)

sample_generation_df = pd.DataFrame(sample_generation_rows)

print("Generated rows:", len(sample_generation_df))
print(
    "Prompt variants:",
    sample_generation_df["prompt_variant"].value_counts().to_dict(),
)
print(
    "Unique questions:",
    sample_generation_df["question_id"].nunique(),
)

sample_generation_df[
    [
        "question_id",
        "prompt_variant",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "latency_ms",
    ]
]

Generated rows: 10
Prompt variants: {'baseline_prompt': 5, 'improved_concise_prompt': 5}
Unique questions: 5


,question_id,prompt_variant,input_tokens,output_tokens,total_tokens,latency_ms
0,question-94f41cc14cb35796d01bf310f052bda1,baseline_prompt,450,97,547,2357.123788
1,question-94f41cc14cb35796d01bf310f052bda1,improved_concise_prompt,355,82,437,3160.081320
2,question-bc48db169054dbcdafcd1d6fbccb6702,baseline_prompt,413,39,452,1403.941396
3,question-bc48db169054dbcdafcd1d6fbccb6702,improved_concise_prompt,318,36,354,1567.233842
4,question-a03266ce0d5eb8986e288a1069e4348b,baseline_prompt,438,79,517,2921.007583
5,question-a03266ce0d5eb8986e288a1069e4348b,improved_concise_prompt,343,75,418,2971.673573
6,question-98c62cabd4f4b0e889e1ba5d8e3423f1,baseline_prompt,499,139,638,2169.693969
7,question-98c62cabd4f4b0e889e1ba5d8e3423f1,improved_concise_prompt,404,66,470,1216.095429
8,question-a20e2f18c0839f18c81e91b9d4c0df95,baseline_prompt,488,158,646,3257.963817
9,question-a20e2f18c0839f18c81e91b9d4c0df95,improved_concise_prompt,393,85,478,4020.173712


In [13]:
generation_summary_df = (
    sample_generation_df
    .groupby("prompt_variant", as_index=False)
    .agg(
        questions=("question_id", "nunique"),
        average_input_tokens=("input_tokens", "mean"),
        average_output_tokens=("output_tokens", "mean"),
        average_total_tokens=("total_tokens", "mean"),
        average_latency_ms=("latency_ms", "mean"),
    )
)

generation_summary_df

,prompt_variant,questions,average_input_tokens,average_output_tokens,average_total_tokens,average_latency_ms
0,baseline_prompt,5,457.6,102.4,560.0,2421.946111
1,improved_concise_prompt,5,362.6,68.8,431.4,2587.051575


In [14]:
print(
    "Missing generated answers:",
    sample_generation_df["answer"].fillna("").str.strip().eq("").sum(),
)

print(
    "Duplicate question/variant pairs:",
    sample_generation_df[
        ["question_id", "prompt_variant"]
    ].duplicated().sum(),
)

Missing generated answers: 0
Duplicate question/variant pairs: 0


In [15]:
## Add a structured LLM judge

from typing import Literal

from pydantic import BaseModel


class JudgeResult(BaseModel):
    relevance: Literal[
        "RELEVANT",
        "PARTLY_RELEVANT",
        "NON_RELEVANT",
    ]
    reasoning: str


JUDGE_INSTRUCTIONS = """
You are evaluating the quality of a generated answer for a
retrieval-augmented question-answering system.

Use the question, reference source excerpt, expected answer, and generated
answer provided by the user.

Classify the generated answer as exactly one of:

- RELEVANT:
  It directly answers the question, is supported by the reference excerpt,
  and does not introduce meaningful unsupported claims.

- PARTLY_RELEVANT:
  It contains the core correct answer but is incomplete, vague, overly broad,
  or includes minor unsupported interpretation.

- NON_RELEVANT:
  It fails to answer the question, contradicts the reference, or relies mainly
  on unsupported information.

Judge factual support against the reference excerpt, not outside knowledge.
Return concise reasoning.
""".strip()


def judge_generated_answer(
    row: pd.Series,
    *,
    model: str = "gpt-4o-mini",
) -> dict:
    """Evaluate one generated answer against its ground-truth source."""
    judge_input = (
        "Question:\n"
        f"{row['question']}\n\n"
        "Reference source excerpt:\n"
        f"{row['source_text']}\n\n"
        "Expected answer:\n"
        f"{row['expected_answer']}\n\n"
        "Generated answer:\n"
        f"{row['answer']}"
    )

    started_at = perf_counter()

    response = baseline_rag.client.responses.parse(
        model=model,
        instructions=JUDGE_INSTRUCTIONS,
        input=judge_input,
        text_format=JudgeResult,
    )

    latency_ms = (perf_counter() - started_at) * 1000

    input_tokens, output_tokens, total_tokens = extract_response_usage(
        response
    )

    parsed = response.output_parsed

    if parsed is None:
        raise RuntimeError("Judge returned no parsed result.")

    return {
        "relevance": parsed.relevance,
        "judge_reasoning": parsed.reasoning,
        "judge_model": model,
        "judge_input_tokens": input_tokens,
        "judge_output_tokens": output_tokens,
        "judge_total_tokens": total_tokens,
        "judge_latency_ms": latency_ms,
        "judge_response_id": getattr(response, "id", None),
    }

In [16]:
## judge on the first two rows only

judge_test_rows = []

for _, row in sample_generation_df.head(2).iterrows():
    judge_result = judge_generated_answer(row)

    judge_test_rows.append(
        {
            "prompt_variant": row["prompt_variant"],
            "answer": row["answer"],
            **judge_result,
        }
    )

judge_test_df = pd.DataFrame(judge_test_rows)

judge_test_df[
    [
        "prompt_variant",
        "relevance",
        "judge_reasoning",
        "judge_total_tokens",
        "judge_latency_ms",
    ]
]

,prompt_variant,relevance,judge_reasoning,judge_total_tokens,judge_latency_ms
0,baseline_prompt,RELEVANT,The generated answer directly addresses how in...,557,3517.488853
1,improved_concise_prompt,RELEVANT,The generated answer accurately reflects the k...,543,2033.364928


In [17]:
## Judge all 10 sample answers

judged_rows = []

for _, row in sample_generation_df.iterrows():
    judge_result = judge_generated_answer(row)

    judged_rows.append(
        {
            **row.to_dict(),
            **judge_result,
        }
    )

sample_judged_df = pd.DataFrame(judged_rows)

print("Judged rows:", len(sample_judged_df))
print(
    "Relevance counts:",
    sample_judged_df["relevance"].value_counts().to_dict(),
)
print(
    "Missing judge results:",
    sample_judged_df["relevance"].isna().sum(),
)

sample_judged_df[
    [
        "question_id",
        "prompt_variant",
        "relevance",
        "judge_total_tokens",
        "judge_latency_ms",
    ]
]

Judged rows: 10
Relevance counts: {'RELEVANT': 10}
Missing judge results: 0


,question_id,prompt_variant,relevance,judge_total_tokens,judge_latency_ms
0,question-94f41cc14cb35796d01bf310f052bda1,baseline_prompt,RELEVANT,570,6546.961823
1,question-94f41cc14cb35796d01bf310f052bda1,improved_concise_prompt,RELEVANT,546,1484.236621
2,question-bc48db169054dbcdafcd1d6fbccb6702,baseline_prompt,RELEVANT,442,1485.875385
3,question-bc48db169054dbcdafcd1d6fbccb6702,improved_concise_prompt,RELEVANT,438,1625.029208
4,question-a03266ce0d5eb8986e288a1069e4348b,baseline_prompt,RELEVANT,524,1005.266718
5,question-a03266ce0d5eb8986e288a1069e4348b,improved_concise_prompt,RELEVANT,543,1857.045382
6,question-98c62cabd4f4b0e889e1ba5d8e3423f1,baseline_prompt,RELEVANT,667,1451.444575
7,question-98c62cabd4f4b0e889e1ba5d8e3423f1,improved_concise_prompt,RELEVANT,589,1480.895067
8,question-a20e2f18c0839f18c81e91b9d4c0df95,baseline_prompt,RELEVANT,673,1547.665650
9,question-a20e2f18c0839f18c81e91b9d4c0df95,improved_concise_prompt,RELEVANT,590,1726.398610


In [18]:
judge_summary_df = (
    sample_judged_df
    .groupby(
        ["prompt_variant", "relevance"],
        as_index=False,
    )
    .agg(
        answers=("question_id", "count"),
        average_generation_tokens=("total_tokens", "mean"),
        average_generation_latency_ms=("latency_ms", "mean"),
        average_judge_tokens=("judge_total_tokens", "mean"),
        average_judge_latency_ms=("judge_latency_ms", "mean"),
    )
)

judge_summary_df

,prompt_variant,relevance,answers,average_generation_tokens,average_generation_latency_ms,average_judge_tokens,average_judge_latency_ms
0,baseline_prompt,RELEVANT,5,560.0,2421.946111,575.2,2407.442830
1,improved_concise_prompt,RELEVANT,5,431.4,2587.051575,541.2,1634.720978


In [19]:
sample_judged_df.loc[
    sample_judged_df["question_id"]
    == "question-a20e2f18c0839f18c81e91b9d4c0df95",
    [
        "prompt_variant",
        "answer",
        "relevance",
        "judge_reasoning",
    ],
].T

,8,9
prompt_variant,baseline_prompt,improved_concise_prompt
answer,To effectively respond to a question that seem...,To effectively respond to a question that seem...
relevance,RELEVANT,RELEVANT
judge_reasoning,The generated answer effectively addresses the...,The generated answer directly addresses the qu...


In [20]:
comparison_rows = sample_judged_df.loc[
    sample_judged_df["question_id"]
    == "question-a20e2f18c0839f18c81e91b9d4c0df95"
]

for _, row in comparison_rows.iterrows():
    print("=" * 100)
    print("PROMPT VARIANT:", row["prompt_variant"])
    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED ANSWER:")
    print(row["expected_answer"])

    print("\nGENERATED ANSWER:")
    print(row["answer"])

    print("\nJUDGE LABEL:")
    print(row["relevance"])

    print("\nJUDGE REASONING:")
    print(row["judge_reasoning"])
    print()

PROMPT VARIANT: baseline_prompt

QUESTION:
How can you effectively respond to a question that seems straightforward but has a deeper underlying concern?

EXPECTED ANSWER:
You should aim to answer the question in a way that addresses the underlying concern, while also validating the reason for the question. This helps to maintain the flow of the conversation.

GENERATED ANSWER:
To effectively respond to a question that seems straightforward but has a deeper underlying concern, you can follow this approach:

1. **Acknowledge the question**: Start by recognizing the question as it is asked.
2. **Provide relevant information**: Offer specific data or insights that pertain to the question.
3. **Explore the underlying concern**: Ask clarifying questions or restate what you think the deeper issue may be to validate the other person's concerns.

Wes Kao illustrates this by saying, “being able to answer a similar question in the direction you think the person is asking about and then validating

In [21]:
## strict judge

JUDGE_INSTRUCTIONS = """
You are evaluating the quality of a generated answer for a
retrieval-augmented question-answering system.

Use only the supplied question, reference source excerpt, expected answer,
and generated answer. Do not use outside knowledge.

Classify the generated answer as exactly one of:

RELEVANT:
- Directly answers the question.
- Preserves the central advice from the reference.
- Contains no meaningful unsupported claims, steps, examples, outcomes,
  or recommendations.
- Minor paraphrasing is acceptable.

PARTLY_RELEVANT:
- Contains the correct core answer, but is incomplete, vague, or unnecessarily
  broad.
- Or adds one or more unsupported interpretations, steps, examples,
  outcomes, or recommendations.
- The unsupported additions do not replace or contradict the core answer.

NON_RELEVANT:
- Fails to provide the core answer.
- Contradicts the reference.
- Relies mainly on unsupported information.
- Answers a different question.

Important:
- Do not reward fluency, detail, structure, or confidence by themselves.
- Extra actionable steps must be supported by the reference excerpt.
- A polished answer with invented guidance should be PARTLY_RELEVANT or
  NON_RELEVANT, depending on how much unsupported content it contains.
- Evaluate factual support and answer precision separately from writing quality.

Return concise reasoning that identifies any unsupported additions.
""".strip()

In [22]:
## rerun judge for 2 answers with the stricter instructions

strict_judge_rows = []

for _, row in comparison_rows.iterrows():
    judge_result = judge_generated_answer(row)

    strict_judge_rows.append(
        {
            "prompt_variant": row["prompt_variant"],
            "relevance": judge_result["relevance"],
            "judge_reasoning": judge_result["judge_reasoning"],
        }
    )

strict_judge_df = pd.DataFrame(strict_judge_rows)

for _, row in strict_judge_df.iterrows():
    print("=" * 100)
    print("PROMPT VARIANT:", row["prompt_variant"])
    print("LABEL:", row["relevance"])
    print("REASONING:")
    print(row["judge_reasoning"])

PROMPT VARIANT: baseline_prompt
LABEL: PARTLY_RELEVANT
REASONING:
The generated answer captures the core concept of addressing the underlying concern and validating the question, aligning with the reference. However, it introduces a structured approach (step-by-step method) that isn't explicitly supported by the excerpt. The addition of suggested steps and the quote from Wes Kao is helpful but leads to a more detailed response than what the reference directly offers. Thus, while it discusses the underlying concern, its structure introduces unsupported elements.
PROMPT VARIANT: improved_concise_prompt
LABEL: RELEVANT
REASONING:
The generated answer directly addresses the question by elaborating on the need to interpret the underlying concern and validate the reason for the question. It preserves the essence of the reference by discussing the importance of context and maintaining the conversation. There are no unsupported additions or claims.


In [25]:
## Rejudge all 10 sample answers with the strict rubric
strict_judged_rows = []

for _, row in sample_generation_df.iterrows():
    judge_result = judge_generated_answer(row)

    strict_judged_rows.append(
        {
            **row.to_dict(),
            **judge_result,
        }
    )

strict_sample_judged_df = pd.DataFrame(strict_judged_rows)

print("Judged rows:", len(strict_sample_judged_df))
print(
    "Overall relevance counts:",
    strict_sample_judged_df["relevance"].value_counts().to_dict(),
)
print(
    "\nCounts by prompt variant:"
)

strict_counts_df = (
    strict_sample_judged_df
    .groupby(["prompt_variant", "relevance"])
    .size()
    .unstack(fill_value=0)
)

strict_counts_df


Judged rows: 10
Overall relevance counts: {'RELEVANT': 9, 'PARTLY_RELEVANT': 1}

Counts by prompt variant:


relevance,PARTLY_RELEVANT,RELEVANT
prompt_variant,,
baseline_prompt,1,4
improved_concise_prompt,0,5


In [26]:
## judging 30 questions 

FULL_EVALUATION_SAMPLE_SIZE = 30
FULL_EVALUATION_RANDOM_SEED = 42

full_evaluation_df = (
    ground_truth_df
    .sample(
        n=FULL_EVALUATION_SAMPLE_SIZE,
        random_state=FULL_EVALUATION_RANDOM_SEED,
    )
    .reset_index(drop=True)
)

print("Evaluation questions:", len(full_evaluation_df))
print(
    "Unique questions:",
    full_evaluation_df["question_id"].nunique(),
)
print(
    "Unique source chunks:",
    full_evaluation_df["chunk_id"].nunique(),
)
print(
    "Unique episodes:",
    full_evaluation_df["episode_id"].nunique(),
)

full_evaluation_df["topic"].value_counts().head(10)

Evaluation questions: 30
Unique questions: 30
Unique source chunks: 30
Unique episodes: 30


topic
product development    3
strategy               3
user research          3
leadership             2
team operations        2
product management     2
career development     1
A/B Testing            1
interviewing           1
M&A                    1
Name: count, dtype: int64

In [27]:
EVALUATION_DIR = PROJECT_ROOT / "data" / "evaluation"

GENERATION_CHECKPOINT_PATH = (
    EVALUATION_DIR
    / "rag-generation-checkpoint.jsonl"
)

JUDGE_CHECKPOINT_PATH = (
    EVALUATION_DIR
    / "rag-judge-checkpoint.jsonl"
)

FINAL_RESULTS_PATH = (
    EVALUATION_DIR
    / "rag-evaluation-results.json"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Generation checkpoint:", GENERATION_CHECKPOINT_PATH)
print("Generation checkpoint exists:", GENERATION_CHECKPOINT_PATH.exists())

print("\nJudge checkpoint:", JUDGE_CHECKPOINT_PATH)
print("Judge checkpoint exists:", JUDGE_CHECKPOINT_PATH.exists())

print("\nFinal results:", FINAL_RESULTS_PATH)

Generation checkpoint: /workspaces/pm-playbook/data/evaluation/rag-generation-checkpoint.jsonl
Generation checkpoint exists: False

Judge checkpoint: /workspaces/pm-playbook/data/evaluation/rag-judge-checkpoint.jsonl
Judge checkpoint exists: False

Final results: /workspaces/pm-playbook/data/evaluation/rag-evaluation-results.json
